
# Clasificación de Carros por Rango de Precio

Este notebook implementa un sistema de clasificación de imágenes usando **Transfer Learning** con EfficientNet.

Objetivo:
- Clasificar carros en:
  - Económico
  - Gama media
  - Lujo

También incluye:
- Evaluación del modelo
- Matriz de confusión
- Predicciones
- Grad-CAM para interpretabilidad



# 1. Instalación de dependencias

Ejecuta esta celda solo si estás trabajando por primera vez.


In [ ]:

# !pip install torch torchvision torchaudio
# !pip install scikit-learn matplotlib seaborn pandas pillow
# !pip install grad-cam



# 2. Importación de librerías

Aquí importamos todas las librerías necesarias:
- PyTorch
- torchvision
- métricas
- visualización
- Grad-CAM


In [ ]:

import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)



# 3. Configuración del proyecto

Aquí definimos:
- Tamaño de batch
- Número de épocas
- Learning rate
- Rutas del dataset


In [ ]:

BATCH_SIZE = 32
IMAGE_SIZE = 224
EPOCHS = 10
LEARNING_RATE = 0.0001

TRAIN_DIR = "dataset/train"
TEST_DIR = "dataset/test"

MODEL_SAVE_PATH = "car_classifier.pth"



# 4. Transformaciones de imágenes

Las imágenes se normalizan y aumentan artificialmente con:
- Rotaciones
- Flip horizontal
- Cambios leves de color

Esto ayuda a evitar overfitting.


In [ ]:

train_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
])

test_transforms = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
])



# 5. Cargar dataset

La estructura del dataset debe verse así:

```text
dataset/
│
├── train/
│   ├── economico/
│   ├── gama_media/
│   └── lujo/
│
└── test/
    ├── economico/
    ├── gama_media/
    └── lujo/
```


In [ ]:

train_dataset = datasets.ImageFolder(
    TRAIN_DIR,
    transform=train_transforms
)

test_dataset = datasets.ImageFolder(
    TEST_DIR,
    transform=test_transforms
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Train images:", len(train_dataset))
print("Test images:", len(test_dataset))

print("Classes:", train_dataset.classes)



# 6. Visualización del dataset

Mostramos algunas imágenes aleatorias del dataset.


In [ ]:

def show_images(dataset, n=6):

    fig, axes = plt.subplots(1, n, figsize=(15,5))

    for i in range(n):

        img, label = dataset[random.randint(0, len(dataset)-1)]

        img = img.permute(1,2,0)

        axes[i].imshow(img)
        axes[i].set_title(dataset.classes[label])
        axes[i].axis("off")

    plt.show()

show_images(train_dataset)



# 7. Modelo preentrenado

Usaremos EfficientNet-B0.

El modelo fue entrenado originalmente en ImageNet.
Nosotros solo cambiamos la capa final para clasificar 3 categorías.


In [ ]:

model = models.efficientnet_b0(pretrained=True)

# Congelar capas convolucionales
for param in model.features.parameters():
    param.requires_grad = False

# Reemplazar clasificador final
in_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(in_features, 3)
)

model = model.to(device)

print(model)



# 8. Función de pérdida y optimizador

Usamos:
- CrossEntropyLoss
- Adam optimizer


In [ ]:

criterion = nn.CrossEntropyLoss()

optimizer = optim.Adam(
    model.parameters(),
    lr=LEARNING_RATE
)



# 9. Entrenamiento

Aquí entrenamos el modelo.


In [ ]:

train_losses = []

for epoch in range(EPOCHS):

    model.train()

    running_loss = 0.0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

    epoch_loss = running_loss / len(train_loader)

    train_losses.append(epoch_loss)

    print(f"Epoch [{epoch+1}/{EPOCHS}] Loss: {epoch_loss:.4f}")



# 10. Guardar modelo

Guardamos el modelo entrenado.


In [ ]:

torch.save(model.state_dict(), MODEL_SAVE_PATH)

print("Modelo guardado correctamente")



# 11. Curva de entrenamiento

Visualizamos la pérdida durante el entrenamiento.


In [ ]:

plt.figure(figsize=(8,5))

plt.plot(train_losses)

plt.title("Training Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")

plt.show()



# 12. Evaluación del modelo

Calculamos:
- Accuracy
- F1-score


In [ ]:

model.eval()

all_preds = []
all_labels = []

with torch.no_grad():

    for images, labels in test_loader:

        images = images.to(device)

        outputs = model(images)

        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

accuracy = accuracy_score(all_labels, all_preds)

f1 = f1_score(
    all_labels,
    all_preds,
    average="macro"
)

print("Accuracy:", accuracy)
print("Macro F1:", f1)



# 13. Reporte de clasificación

Mostramos precisión, recall y F1-score por clase.


In [ ]:

print(classification_report(
    all_labels,
    all_preds,
    target_names=train_dataset.classes
))



# 14. Matriz de confusión

Permite visualizar los errores del modelo.


In [ ]:

cm = confusion_matrix(all_labels, all_preds)

plt.figure(figsize=(7,5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    xticklabels=train_dataset.classes,
    yticklabels=train_dataset.classes
)

plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")

plt.show()



# 15. Predicciones aleatorias

Mostramos imágenes y predicciones del modelo.


In [ ]:

def predict_random_images(dataset, model, n=5):

    model.eval()

    fig, axes = plt.subplots(1, n, figsize=(15,5))

    for i in range(n):

        idx = random.randint(0, len(dataset)-1)

        image, label = dataset[idx]

        input_tensor = image.unsqueeze(0).to(device)

        with torch.no_grad():

            output = model(input_tensor)

            pred = torch.argmax(output, 1).item()

        img = image.permute(1,2,0)

        axes[i].imshow(img)

        axes[i].set_title(
            f"P: {dataset.classes[pred]}\nT: {dataset.classes[label]}"
        )

        axes[i].axis("off")

    plt.show()

predict_random_images(test_dataset, model)



# 16. Grad-CAM

Grad-CAM permite visualizar qué partes de la imagen utilizó el modelo para tomar la decisión.

Esto es importante para interpretabilidad.


In [ ]:

target_layers = [model.features[-1]]

cam = GradCAM(
    model=model,
    target_layers=target_layers
)

image, label = test_dataset[0]

input_tensor = image.unsqueeze(0).to(device)

targets = [ClassifierOutputTarget(label)]

grayscale_cam = cam(
    input_tensor=input_tensor,
    targets=targets
)

grayscale_cam = grayscale_cam[0]

rgb_img = image.permute(1,2,0).numpy()

visualization = show_cam_on_image(
    rgb_img,
    grayscale_cam,
    use_rgb=True
)

plt.figure(figsize=(10,5))

plt.subplot(1,2,1)
plt.imshow(rgb_img)
plt.title("Original")
plt.axis("off")

plt.subplot(1,2,2)
plt.imshow(visualization)
plt.title("Grad-CAM")
plt.axis("off")

plt.show()



# 17. Predicción de imágenes externas

Puedes probar imágenes nuevas.


In [ ]:

def predict_image(image_path):

    model.eval()

    image = Image.open(image_path).convert("RGB")

    transform = transforms.Compose([
        transforms.Resize((224,224)),
        transforms.ToTensor()
    ])

    image_tensor = transform(image).unsqueeze(0).to(device)

    with torch.no_grad():

        output = model(image_tensor)

        pred = torch.argmax(output,1).item()

    plt.imshow(image)

    plt.title(
        f"Prediction: {train_dataset.classes[pred]}"
    )

    plt.axis("off")

    plt.show()

# Ejemplo:
# predict_image("mi_carro.jpg")
